In [1]:
import pandas as pd
import psycopg2

sheet_curr = '1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk'
sheet_archive = '1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE'
gid_matches = '2141931777'
gid_deck = '590005429'

# MATCH_ID      = 11000000000
# EVENT_ID      = 12000000000
# DECK_ID       = 13000000000
# EVENT_TYPE_ID = 14000000000
# LOAD_RPT_ID   = 15000000000
# REJ_ID        = 16000000000

In [2]:
with open("credentials.txt", "r") as file:
    credentials = [line.strip() for line in file]

In [3]:
def conn(query,vars=()):
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        if len(vars) > 0:
            cursor.execute(query,vars)
        else:
            cursor.execute(query)

        conn.commit()
    except psycopg2.Error as e:
        print('Error:', e)
    finally:
        if conn:
            cursor.close()
            conn.close()

In [4]:
def get_deck_id(format, archetype, subarchetype, credentials=credentials):
    query = """
        SELECT DECK_ID FROM VALID_DECKS 
        WHERE FORMAT = %s AND ARCHETYPE = %s AND SUBARCHETYPE = %s;
    """
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        cursor.execute(query, (format, archetype, subarchetype))
        if cursor.fetchone() is None:
            cursor.execute(query, ('VINTAGE', 'NA', 'NA'))
    except psycopg2.Error as e:
        print('Error:', e)
    finally:
        if conn:
            cursor.close()
            conn.close()
    return cursor.fetchone()[0]

In [ ]:
def parse_matchup_sheet(sheet,gid,match_id_start=1000000000,event_id_start=1000000):
    sheet_url = f'https://docs.google.com/spreadsheets/d/{sheet}/export?format=csv&gid={gid}'
    df = pd.read_csv(sheet_url)

    # Rename columns.
    df.columns = ['P1','P2','P1_WINS','P2_WINS','WINNER1','WINNER2','P1_ARCH','P2_ARCH','P1_SUBARCH','P2_SUBARCH','P1_NOTE','P2_NOTE','EVENT_DATE','EVENT_TYPE']

    # Replace null values with 'NA' string.
    df.fillna({'P1_ARCH':'NA','P2_ARCH':'NA','P1_SUBARCH':'NA','P2_SUBARCH':'NA','P1_NOTE':'NA','P2_NOTE':'NA'},inplace=True)

    # Strip whitespace from player/deck names.
    df.P1 = df.P1.str.strip()
    df.P2 = df.P2.str.strip()
    df.P1_ARCH = df.P1_ARCH.str.strip().str.upper()
    df.P2_ARCH = df.P2_ARCH.str.strip().str.upper()
    df.P1_SUBARCH = df.P1_SUBARCH.str.strip().str.upper()
    df.P2_SUBARCH = df.P2_SUBARCH.str.strip().str.upper()
    df.P1_NOTE = df.P1_NOTE.str.strip().str.upper()
    df.P2_NOTE = df.P2_NOTE.str.strip().str.upper()

    # Format EVENT_TYPE values.
    df['EVENT_TYPE'] = df['EVENT_TYPE'].str.upper().str.strip()

    # Format No Show deck name values.
    for index, row in df.iterrows():
        if row['P1_SUBARCH'].strip().upper() == 'NO SHOW':
            df.at[index,'P1_SUBARCH'] = 'NO SHOW'
        if row['P2_SUBARCH'].strip().upper() == 'NO SHOW':
            df.at[index,'P2_SUBARCH'] = 'NO SHOW'

    # Format date column.
    df.EVENT_DATE = pd.to_datetime(df.EVENT_DATE,yearfirst=False,format='mixed')

    # Adding Event_IDs.
    count = event_id_start
    df['EVENT_ID'] = 0
    for index, row in reversed(list(df.iterrows())):
        df.at[index,'EVENT_ID'] = count
        if pd.notna(row['EVENT_TYPE']):
            count += 1

    # Handle empty EVENT_TYPE/EVENT_DATE values by forward-filling.
    df['EVENT_TYPE'] = df['EVENT_TYPE'].ffill()
    df['EVENT_DATE'] = df['EVENT_DATE'].ffill()

    # Ignore events with incomplete data.
    events_to_ignore = [1000007]
    df = df[~df.EVENT_ID.isin(events_to_ignore)]

    # Replace Winner1/2 columns with single Match_Winner column.
    df['MATCH_WINNER'] = df.apply(lambda row: 'P1' if ((row['WINNER1'] == 1) & (row['WINNER2'] == 0)) else ('P2' if ((row['WINNER1'] == 0) & (row['WINNER2'] == 1)) else 'NA'), axis=1)
    df.drop(columns=['WINNER1','WINNER2'],inplace=True)

    # EVENT_ID 1000067 should be OTHER
    df.loc[df['EVENT_ID'] == 1000067,'EVENT_TYPE'] = 'OTHER'

    # Convert P1/P2_WINS from float to int.
    df['P1_WINS'] = df['P1_WINS'].astype(int)
    df['P2_WINS'] = df['P2_WINS'].astype(int)

    # Abstract out Event info into its own table.
    df_events = df.groupby(['EVENT_ID','EVENT_DATE']).agg({'EVENT_TYPE':'last'}).reset_index()

    # Calculate MATCH_IDs for each pair of rows that apply to the same match.
    df['match_key'] = df.apply(lambda row: frozenset([row['P1'],row['P2'],row['EVENT_ID']]), axis=1)
    df = df.reset_index()
    df = df.sort_values(by=['match_key','index'])
    df["group_id"] = df.groupby(['match_key']).cumcount() // 2
    df["MATCH_ID"] = (df.groupby(['match_key','group_id']).ngroup() + match_id_start)
    df = df.sort_values(by=['index'])
    df = df.drop(columns=['match_key','group_id','index'])

    return df, df_events

In [7]:
def parse_matchup_sheet2(sheet,gid,match_id_start=1000000000,event_id_start=1000000):
    sheet_url = f'https://docs.google.com/spreadsheets/d/{sheet}/export?format=csv&gid={gid}'
    df = pd.read_csv(sheet_url)

    # Rename columns.
    df.columns = ['P1','P2','P1_WINS','P2_WINS','WINNER1','WINNER2','P1_ARCH','P2_ARCH','P1_SUBARCH','P2_SUBARCH','P1_NOTE','P2_NOTE','EVENT_DATE','EVENT_TYPE']

    # Replace null values with 'NA' string.
    df.fillna({'P1_ARCH':'NA','P2_ARCH':'NA','P1_SUBARCH':'NA','P2_SUBARCH':'NA','P1_NOTE':'NA','P2_NOTE':'NA'},inplace=True)

    # Strip whitespace from player/deck names.
    df.P1 = df.P1.str.strip()
    df.P2 = df.P2.str.strip()
    df.P1_ARCH = df.P1_ARCH.str.strip().str.upper()
    df.P2_ARCH = df.P2_ARCH.str.strip().str.upper()
    df.P1_SUBARCH = df.P1_SUBARCH.str.strip().str.upper()
    df.P2_SUBARCH = df.P2_SUBARCH.str.strip().str.upper()
    df.P1_NOTE = df.P1_NOTE.str.strip().str.upper()
    df.P2_NOTE = df.P2_NOTE.str.strip().str.upper()

    # Format EVENT_TYPE values.
    df['EVENT_TYPE'] = df['EVENT_TYPE'].str.upper().str.strip()

    # Format No Show deck name values.
    for index, row in df.iterrows():
        if row['P1_SUBARCH'].strip().upper() == 'NO SHOW':
            df.at[index,'P1_SUBARCH'] = 'NO SHOW'
        if row['P2_SUBARCH'].strip().upper() == 'NO SHOW':
            df.at[index,'P2_SUBARCH'] = 'NO SHOW'

    # Format date column.
    df.EVENT_DATE = pd.to_datetime(df.EVENT_DATE,yearfirst=False,format='mixed')

    # Adding Event_IDs.
    count = event_id_start
    df['EVENT_ID'] = 0
    for index, row in reversed(list(df.iterrows())):
        df.at[index,'EVENT_ID'] = count
        if pd.notna(row['EVENT_TYPE']):
            count += 1

    # Handle empty EVENT_TYPE/EVENT_DATE values by forward-filling.
    df['EVENT_TYPE'] = df['EVENT_TYPE'].ffill()
    df['EVENT_DATE'] = df['EVENT_DATE'].ffill()

    return df

In [19]:
df_matches[df_matches['P1_WINS'].isnull()].groupby(['EVENT_ID']).agg({'P1':'count'}).reset_index()['EVENT_ID'].tolist()

[1000006]

In [28]:
df_matches.groupby(['EVENT_ID']).agg({'P1':'nunique','EVENT_TYPE':'last','EVENT_DATE':'last'}).sort_values(by=['P1'],ascending=False).reset_index()

,EVENT_ID,P1,EVENT_TYPE,EVENT_DATE
0,1000066,199,CHALLENGE,2024-12-28
1,1000085,183,CHALLENGE,2025-02-01
2,1000002,105,CHALLENGE,2024-08-31
3,1000068,62,CHALLENGE,2025-01-02
4,1000061,59,CHALLENGE,2024-12-21
...,...,...,...,...
82,1000060,32,CHALLENGE,2024-12-20
83,1000082,32,CHALLENGE,2025-01-26
84,1000024,32,CHALLENGE,2024-10-13
85,1000058,32,CHALLENGE,2024-12-15


: 

In [22]:
df_matches[df_matches['EVENT_ID'] == 1000066]

,P1,P2,P1_WINS,P2_WINS,WINNER1,WINNER2,P1_ARCH,P2_ARCH,P1_SUBARCH,P2_SUBARCH,P1_NOTE,P2_NOTE,EVENT_DATE,EVENT_TYPE,EVENT_ID
5452,Montolio,billsive,2.0,0.0,1,0,LURRUS,LURRUS,ESPER LURRUS CONTROL,LURRUS PO,NA,NA,2024-12-28,CHALLENGE,1000066
5453,billsive,Montolio,0.0,2.0,0,1,LURRUS,LURRUS,LURRUS PO,ESPER LURRUS CONTROL,NA,NA,2024-12-28,CHALLENGE,1000066
5454,Montolio,duke12,2.0,0.0,1,0,LURRUS,LURRUS,ESPER LURRUS CONTROL,ESPER LURRUS CONTROL,NA,NA,2024-12-28,CHALLENGE,1000066
5455,billsive,matiaskm,2.0,1.0,1,0,LURRUS,LURRUS,LURRUS PO,UB LURRUS CONTROL,NA,NA,2024-12-28,CHALLENGE,1000066
5456,matiaskm,billsive,1.0,2.0,0,1,LURRUS,LURRUS,UB LURRUS CONTROL,LURRUS PO,NA,NA,2024-12-28,CHALLENGE,1000066
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6647,oosunq,magiclaszlo,1.0,2.0,0,1,SHOPS,LURRUS,JEWEL SHOPS,ESPER LURRUS CONTROL,NA,NA,2024-12-28,CHALLENGE,1000066
6648,scalo94,musasabi,0.0,2.0,0,1,LURRUS,AGGRO,LURRUS PO,INITIATIVE,NA,NA,2024-12-28,CHALLENGE,1000066
6649,KingHairy,Varo,1.0,2.0,0,1,COMBO,LURRUS,BREACH,ESPER LURRUS CONTROL,NA,NA,2024-12-28,CHALLENGE,1000066
6650,discoverN,thedeck84,1.0,2.0,0,1,COMBO,COMBO,DOOMSDAY,BREACH,NA,NA,2024-12-28,CHALLENGE,1000066


In [8]:
df_matches = parse_matchup_sheet2(sheet_curr,gid_matches)
df_matches

,P1,P2,P1_WINS,P2_WINS,WINNER1,WINNER2,P1_ARCH,P2_ARCH,P1_SUBARCH,P2_SUBARCH,P1_NOTE,P2_NOTE,EVENT_DATE,EVENT_TYPE,EVENT_ID
0,crK,Folero,2.0,1.0,1,0,SHOPS,LURRUS,OTHER SHOPS,UB LURRUS CONTROL,FLESHRAKER,NA,2025-02-02,CHALLENGE,1000086
1,Folero,crK,1.0,2.0,0,1,LURRUS,SHOPS,UB LURRUS CONTROL,OTHER SHOPS,NA,FLESHRAKER,2025-02-02,CHALLENGE,1000086
2,crK,Redhotphil87,2.0,0.0,1,0,SHOPS,COMBO,OTHER SHOPS,DOOMSDAY,FLESHRAKER,NA,2025-02-02,CHALLENGE,1000086
3,Folero,Tsubasa_Cat,2.0,1.0,1,0,LURRUS,COMBO,UB LURRUS CONTROL,DOOMSDAY,NA,NA,2025-02-02,CHALLENGE,1000086
4,Tsubasa_Cat,Folero,1.0,2.0,0,1,COMBO,LURRUS,DOOMSDAY,UB LURRUS CONTROL,NA,NA,2025-02-02,CHALLENGE,1000086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22727,2plus2isfive,Zenith777,1.0,2.0,0,1,OATH,BAZAAR,OATH,DREDGE,NA,NA,2024-08-29,CHALLENGE,1000000
22728,s063,unluckymonkey,0.0,2.0,0,1,COMBO,LURRUS,DOOMSDAY,LURRUS DRS,NA,NA,2024-08-29,CHALLENGE,1000000
22729,OrnatePuzzles,TonyScapone,1.0,2.0,0,1,SHOPS,SHOPS,SPHERE SHOPS,JEWEL SHOPS,NA,NA,2024-08-29,CHALLENGE,1000000
22730,Ale_Mtg,AFX,0.0,2.0,0,1,FAIR BLUE,BAZAAR,BUG,COUNTERVINE,NA,NA,2024-08-29,CHALLENGE,1000000


In [ ]:
def merge_matches_codes(df_matches,df_valid_decks):
    df = pd.merge(left=df_matches,right=df_valid_decks,left_on=['P1_ARCH','P1_SUBARCH'],right_on=['ARCHETYPE','SUBARCHETYPE'],how='left')
    df.rename(columns={'DECK_ID':'P1_DECK_ID'}, inplace=True)
    df = pd.merge(left=df,right=df_valid_decks,left_on=['P2_ARCH','P2_SUBARCH'],right_on=['ARCHETYPE','SUBARCHETYPE'],how='left')
    df.rename(columns={'DECK_ID':'P2_DECK_ID'}, inplace=True)
    return df[['MATCH_ID','P1','P2','P1_WINS','P2_WINS','MATCH_WINNER','P1_DECK_ID','P2_DECK_ID','P1_NOTE','P2_NOTE','EVENT_ID']]

In [ ]:
def merge_events_codes(df_events,df_valid_events):
    df = pd.merge(left=df_events,right=df_valid_events,left_on=['EVENT_TYPE'],right_on=['EVENT_TYPE'],how='left')
    return df[['EVENT_ID','EVENT_DATE','EVENT_TYPE_ID']]

In [ ]:
def insert(df_valid_decks=None, df_valid_event_types=None, df_matches=None, df_events=None, credentials=credentials):
    valid_decks_query = """
        INSERT INTO VALID_DECKS (FORMAT, ARCHETYPE, SUBARCHETYPE, DECK_ID)
        VALUES (%s, %s, %s, %s)
    """
    valid_event_types_query = """
        INSERT INTO VALID_EVENT_TYPES (FORMAT, EVENT_TYPE, EVENT_TYPE_ID)
        VALUES (%s, %s, %s)
    """
    events_query = """
        INSERT INTO EVENTS (EVENT_ID, EVENT_DATE, EVENT_TYPE_ID)
        VALUES (%s, %s, %s)
    """
    matches_query = """
        INSERT INTO MATCHES (MATCH_ID, P1, P2, P1_WINS, P2_WINS, MATCH_WINNER, P1_DECK_ID, P2_DECK_ID, P1_NOTE, P2_NOTE, EVENT_ID)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        # Insert valid_decks
        if df_valid_decks is not None:
            values_list = [(row['FORMAT'], row['ARCHETYPE'], row['SUBARCHETYPE'], row['DECK_ID']) for index, row in df_valid_decks.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(valid_decks_query, values)
                except Exception as e:
                    print(f"Error inserting row into VALID_DECKS: {values} | Error: {e}")
                    continue  # Skip the row and continue with the next one
            conn.commit()

        # Insert valid_event_types
        if df_valid_event_types is not None:
            values_list = [(row['FORMAT'], row['EVENT_TYPE'], row['EVENT_TYPE_ID']) for index, row in df_valid_event_types.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(valid_event_types_query, values)
                except Exception as e:
                    print(f"Error inserting row into VALID_EVENT_TYPES: {values} | Error: {e}")
                    continue
            conn.commit()

        # Insert events
        if df_events is not None:
            values_list = [(row['EVENT_ID'], row['EVENT_DATE'], row['EVENT_TYPE_ID']) for index, row in df_events.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(events_query, values)
                except Exception as e:
                    print(f"Error inserting row into EVENTS: {values} | Error: {e}")
                    continue
            conn.commit()

        # Insert matches
        if df_matches is not None:
            values_list = [(row['MATCH_ID'], row['P1'], row['P2'], row['P1_WINS'], row['P2_WINS'],
                            row['MATCH_WINNER'], row['P1_DECK_ID'], row['P2_DECK_ID'], row['P1_NOTE'], row['P2_NOTE'], row['EVENT_ID']) 
                           for index, row in df_matches.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(matches_query, values)
                except Exception as e:
                    print(f"Error inserting row into MATCHES: {values} | Error: {e}")
                    continue
            conn.commit()

    except Exception as e:
        print(f"Error occurred while loading data: {e}")
        conn.rollback()

    finally:
        if conn:
            cursor.close()
            conn.close()

In [ ]:
df_matches, df_events = parse_matchup_sheet(sheet_curr,gid_matches)
df_matches = merge_matches_codes(df_matches,df_valid_decks)
df_events = merge_events_codes(df_events,df_valid_event_types)

In [54]:
def delete_table(TABLE):
    query = f'DROP TABLE IF EXISTS {TABLE} CASCADE'
    conn(query)

In [ ]:
insert(df_valid_decks,df_valid_event_types,df_events,df_matches)

### Test functions.

In [79]:
df_valid_decks.iloc[10,2] = 'POOPSDAY2'
df_valid_event_types.iloc[2,1] = 'POOPLIM2'